# Pakete importieren, csv readen
### Inhalt
-  __Brands__: ist die Marke eines Produktes
-  __Labels__: sind Hinweise/Anmerkungen auf einem Produkt wie zum Beispiel glutenfrei, vegetarisch, made in, gute Proteinquelle usw.
-  __Countries__: Wo ein Eintrag verkauft wurde
-  __Food groups__: Kategorie eines Eintrags
-  __main_category__: Gibt Hauptkategorie eines Eintrags.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import re
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Verhindert den Zeilenumbruch auf mehrere Zeilen
pd.set_option('display.expand_frame_repr', False)


In [ ]:
data = pd.read_csv("openfoodfacts.csv", sep="\t", low_memory=False, on_bad_lines="skip")

In [ ]:
df = data.copy()

# Spaltenfilter und strip
df nur mit meinen Spalten

In [ ]:
meine_spalten = ["brands", "brands_tags", "brands_en", 
                 "labels", "labels_tags", "labels_en", 
                 "countries", "countries_tags", "countries_en", 
                 "food_groups", "food_groups_tags", "food_groups_en", 
                 "main_category", "main_category_en", 
                 "pnns_groups_1", "pnns_groups_2",
                 "brand_owner",
                 "popularity_tags",
                 "origins", "origins_tags", "origins_en",
                 "purchase_places",
                 "cities_tags",
                 "no_nutrition_data"]
df = df[meine_spalten]
df[meine_spalten] = df[meine_spalten].apply(lambda x: x.strip() if isinstance(x, str) else x)

# Funktionen
-  completeness_spalte(df, spalte) # gibt an wie viel Prozent der Werte in Spalte nicht NaN sind
-  clean_brands_pandas(df, top_n=500) # standardisiert und behält die top 500 Marken
-  top_x_percent_spalte(df, spalte, prozent)
-  delete_rows_with_words_in_spalte(df, spalte, woerter) # woerter als regulärer Ausdruck r'\bx\b
-  contains_x_in_spalte(df, spalte, x) #gibt df das x in spalte stehen hat
-  top_x_spalte(df, spalte, x)
-  bottom_x_spalte(df, spalte, x)
-  entferne_fehleintrage(eintrag, fehler_wort_set) #eintrag muss nicht als parameter gegeben werden
-  uebersetze_string_laender(text, land_woerterbuch)

In [ ]:
def completeness_spalte(df, spalte):
    return df[spalte].notna().mean()*100

In [ ]:
# ==========================================
# LÖSUNG MIT PANDAS
# ==========================================
def clean_brands_pandas(df, top_n=500):
    # 1. Text normalisieren: Kleinbuchstaben & Whitespaces entfernen
    df['brands_clean'] = df['brands_en'].str.lower().str.strip()
    
    # 2. Akzente entfernen (é -> e, ä -> a)
    # Wichtig: Benötigt oft das 'unidecode' Paket oder Pandas string normalize
    df['brands_clean'] = df['brands_clean'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
    
    # 3. Satzzeichen und Firmenzusätze (GmbH, Inc, SA) mit Regex entfernen
    # Ersetzt alles was kein Buchstabe oder Zahl ist mit einem Leerzeichen
    df['brands_clean'] = df['brands_clean'].str.replace(r'[^a-z0-9\s]', ' ', regex=True)
    # Entfernt gängige Rechtsformen
    df['brands_clean'] = df['brands_clean'].str.replace(r'\b(gmbh|inc|ltd|sa|co|kg|ag)\b', '', regex=True)
    
    # Letzte doppelte Leerzeichen entfernen, die durch das Löschen entstanden sind
    df['brands_clean'] = df['brands_clean'].str.replace(r'\s+', ' ', regex=True).str.strip()
    
    # 4. THRESHOLDING (Die Top N behalten, den Rest zu 'other' machen)
    # Finde die N häufigsten Marken
    top_brands = df['brands_clean'].value_counts().nlargest(top_n).index
    
    # Alle Marken, die NICHT in den Top Brands sind, werden zu 'other'
    df.loc[~df['brands_clean'].isin(top_brands), 'brands_clean'] = 'other'
    
    return df

In [ ]:
# prozent in dezimal angeben, z.B. 0.90 für 90%
# Funktion, die die kumulierten Prozente berechnet und die Einträge zurückgibt, die bis zu einem bestimmten Prozentwert beitragen
def top_x_percent_spalte(df, spalte, prozent, with_lower_case=True):
    if with_lower_case:
        counts = df[spalte].str.split(",").explode().str.strip().str.lower().value_counts()
    else:
        counts = df[spalte].str.split(",").explode().str.strip().value_counts()
    cum_percentage = counts.cumsum() / counts.sum()
    return cum_percentage[cum_percentage <= prozent]

In [ ]:
# Löscht die komplette Zeile, wenn "ice cream", "test" ODER "unknown" als eigenständiges Wort vorkommt
#fehler_woerter = r'\b(?:Saba|Snacks|it:ringo-gelato-cacao)\b'
# df = df[~df['countries_en'].str.contains(fehler_woerter, na=False, case=False, regex=True)]

def delete_rows_with_words_in_spalte(df, spalte, woerter):
    return df[~df[spalte].astype(str).str.contains(woerter, na=False, case=False, regex=True)]


In [ ]:
def contains_x_in_spalte(df, spalte, x):
    return df[df[spalte].str.contains(x, na=False, case=False, regex=True)]

In [ ]:
def top_x_spalte(df, spalte, x):
    return df[spalte].str.split(",").explode().str.strip().value_counts().head(x)

def bottom_x_spalte(df, spalte, x):
    return df[spalte].str.split(",").explode().str.strip().value_counts().tail(x)

In [ ]:
def entferne_fehlereintrage(eintrag, fehler_wort_set: set={}):
    # Leere Zellen (NaN) einfach überspringen
    if pd.isna(eintrag):
        return eintrag
        
    # Wenn der Eintrag als kommagetrennter String vorliegt
    if isinstance(eintrag, str):
        # 1. Am Komma trennen und Leerzeichen entfernen
        laender = [land.strip() for land in eintrag.split(',')]
        
        # 2. Alles behalten, was NICHT das Fehlerwort ist (Groß-/Kleinschreibung wird ignoriert)
        bereinigt = [land for land in laender if land.lower() not in fehler_wort_set]
        
        # 3. Wieder mit einem sauberen Komma zusammenfügen
        return ','.join(bereinigt)
        
    # Falls die Einträge im DataFrame durch vorherige Schritte doch als Listen vorliegen
    elif isinstance(eintrag, list):
        return [land for land in eintrag if land.strip().lower() not in fehler_wort_set]
        
    return eintrag

# Die Funktion auf die Spalte anwenden
# df['countries_en'] = df['countries_en'].apply(entferne_fehlereintrag)

In [ ]:
def uebersetze_mit_dict(text, woerterbuch):
    """
    Trennt einen String am Komma, wendet ein Wörterbuch an,
    entfernt None-Werte und fügt ihn wieder als String zusammen.
    """
    # Wenn die Zelle komplett leer (NaN) ist, mach nichts
    if pd.isna(text):
        return text
        
    # 1. Am Komma trennen und jedes Element bereinigen (strip + lowercase)
    # 2. Übersetzen: Wenn nicht im Dict, behalte den sauberen Originalwert
    elemente = [woerterbuch.get(land.strip().lower(), land.strip().lower()) for land in str(text).split(',')]
    
    # 3. Das WICHTIGSTE: Alle 'None' Werte herausfiltern, die das .join() zum Abstürzen bringen
    bereinigt = [land for land in elemente if land is not None]
    
    # 4. Wenn die Liste jetzt leer ist (z.B. weil nur "unknown" in der Zelle stand), gib NaN zurück
    if not bereinigt:
        return np.nan
        
    # 5. Als String wieder zusammenfügen (mit Komma und Leerzeichen für die Lesbarkeit)
    return ', '.join(bereinigt)

# Countries
Hauptsächlich Umgang mit countries_en Spalte:
-  Auffüllen mit anderen Spalten
-  Löschen von countries und countries_tags
-  Übersetzer mit paar Einträgen

In [ ]:
countries_spalten = ["countries", "countries_tags", "countries_en"]
countries = df[countries_spalten]
countries.sample(20)

## Bereinigung

### Spalte auffüllen und andere löschen
Ergebnis: 5 Einträge wurden hinzugefügt

In [ ]:
#Prozent und Anzahl der Spalte vor Bearbeitung
countries_en_completeness = str(df["countries_en"].notna().mean()*100)
countries_en_before = df["countries_en"].notna().sum()

print("Vor Zusammenführen war Spalte zu " + countries_en_completeness + " gefüllt")

#countries_en mit countries_tags auffüllen falls leer, mit countries auffüllen falls auch leer
df["countries_en"] = (df["countries_en"]
                                .fillna(df["countries_tags"]
                                .fillna(df["countries"])))

#Prozent und Anzahl der Spalte nach Bearbeitung
countries_en_completeness = str(df["countries_en"].notna().mean()*100)
countries_en_after = df["countries_en"].notna().sum()

countries_added = str(countries_en_after - countries_en_before)

print("Nach Zusammenführen ist Spalte zu " + countries_en_completeness + " gefüllt")
print("Es wurden " + countries_added + " Einträge hinzugefügt")

#countries und countries_tags löschen
df = df.drop(columns=["countries", "countries_tags"])

### Länder übersetzen

In [ ]:
woerterbuch = {
    # --- Deine bisherigen Einträge ---
    'Australien': 'Australia',
    'australien': 'Australia',
    'Deutschland': 'Germany', 
    'Germania': 'Germany', 
    'de:allemagne': 'Germany', 
    'fr:deutschland': 'Germany',
    'Frankreich': 'France', 
    'فرنسا': 'France',
    'fr:italien': 'Italy',
    'Japon': 'Japan',
    'Belgio': 'Belgium',
    'it:paesi-ue': 'EU', 
    'ca:union-europea': 'EU',
    'Suede': 'Sweden',
    'Svizzera': 'Switzerland', 'Schweiz': 'Switzerland',
    'Polonia': 'Poland',
    'صنعاء': 'Yemen',
    'عدن،صنعاء': 'Yemen',
    'Brazil-en-france': 'Brazil, France',
    'de:great-britain': 'United Kingdom',
    'Saudi': 'Saudi Arabia',
    
    # --- Neue Länderzuordnungen ---
    'Irland': 'Ireland',
    'Estados-unidos': 'United States',
    'كندا': 'Canada',
    'British-columbia-canada': 'Canada',
    'Vancouver-bc-canada': 'Canada',
    'Usa-new-york-only': 'United States',
    'St-vincent-and-grenadines': 'Saint Vincent and the Grenadines',
    'فلسطين': 'Palestine',
    'Svalbard and Jan Mayen': 'Svalbard and Jan Mayen',
    'Inde': 'India',
    'Pologne': 'Poland',
    'Roumanie': 'Romania',
    'Turquie': 'Turkey',
    'Algerie': 'Algeria',
    'Polynesie-francaise': 'French Polynesia',
    'de:スペイン': 'Spain',          # Japanisches Katakana für Spanien
    'de:deitschland': 'Germany',   # Dialekt/Tippfehler
    'de:německo': 'Germany',       # Tschechisch für Deutschland
    'nl:roemenie': 'Romania',
    'bg:литвс': 'Lithuania',       # Tippfehler für Litauen (Литва)
    'si:ශ්\u200dරී-ලංකාව': 'Sri Lanka',
    'nl:moldavie': 'Moldova',
    'it:giappo': 'Japan',          # Italienische Abkürzung
    'Wales': 'United Kingdom',     # Teilregion, standardisiert meist UK
    'el:ελλαδα': 'Greece',
    'Saint-Barthélemy': 'Saint Barthélemy',
    'ca:franca': 'France',
    'Eswatini': 'Eswatini',
    'Central-africa': 'Central African Republic',
    'ar:تونس': 'Tunisia',
    'ar:jorsan': 'Jordan',         # Eindeutiger Tippfehler
    'Saba': 'Saba'
}

In [ ]:
# Auf die Spalte anwenden
df['countries_en'] = df['countries_en'].apply(lambda x: uebersetze_mit_dict(x, woerterbuch))

### Komplette Zeile löschen

In [ ]:
# Löscht die komplette Zeile, wenn "ice cream", "test" ODER "unknown" als eigenständiges Wort vorkommt
fehler_woerter = r'\b(?:Saba|Snacks|it:ringo-gelato-cacao)\b'

df = df[~df['countries_en'].str.contains(fehler_woerter, na=False, case=False, regex=True)]

### Länder anzahl anzeigen

In [ ]:
# Alles in einem Schritt: Trennen -> Explodieren -> Säubern -> Zählen
laender_counts = (
    df['countries_en']
    .dropna()             # 1. Leere Zellen ignorieren
    .str.split(',')       # 2. Am Komma in Listen aufteilen
    .explode()            # 3. Listen direkt auflösen (passiert nur im Hintergrund)
    .str.strip()          # 4. Störende Leerzeichen vor/nach Ländern entfernen
    .value_counts()       # 5. Alle Länder zählen
)

# Jetzt aus diesem Ergebnis nur die filtern, die exakt 1 mal vorkommen
anzahl = len(laender_counts[laender_counts == 1])
laender_liste = laender_counts[laender_counts <= 50].index.to_list()
print(laender_liste)

print(f"Es gibt {anzahl} Länder, die exakt ein einziges Mal vorkommen.")

### Anzeigen welche Länder in bis zu 90% der Einträge sind
Ergebnis: 90% des Datensatzes hat keine fehlerhaften Werte, doppelt Einträge werden im Sinne späterer Analysen nicht gesplittet, da sonst Analysen in anderen Bereichen erschwert werden

In [ ]:
# 1. Zählen und sortieren (Pandas macht das bei value_counts standardmäßig)
counts = df["countries_en"].str.split(",").explode().str.strip().value_counts()

# 2. & 3. Kumulierte Prozente berechnen
cum_percentage = counts.cumsum() / counts.sum()

# 4. Den Filter anwenden (Alle Länder, deren Wert <= 0.90 ist)
top_90_percent_countries = cum_percentage[cum_percentage <= 0.90]

# Das Ergebnis anzeigen
print(top_90_percent_countries)

# Den Haupt-Datensatz filtern:
df_final = df[df["countries_en"].isin(top_90_percent_countries.index)]

In [ ]:
top_50_laender = top_x_spalte(df, "countries_en", 50)
top_50_laender

In [ ]:
bottom_10_laender = bottom_x_spalte(df, "countries_en", 10)
bottom_10_laender

# Labels
Beinhalter Nutriscore Grades manchmal
-  Labels_en Spalte bis 90 Prozent bereinigt
-  Aufgefüllt mir _tags und dann labels, falls leer

### Labelsspalte anschauen

In [ ]:
labels_spalten = ["labels", "labels_tags", "labels_en"]
all_labels = df[labels_spalten].sample(20)
all_labels

### Labelsspalte auffüllen und löschen
-   38 Einträge dazu gekommen
-  27,2251% -> 27,2259% (+ 0,0008%)

In [ ]:
#Prozent und Anzahl der Spalte vor Bearbeitung
labels_en_completeness = str(df["labels_en"].notna().mean()*100)
labels_en_before = df["labels_en"].notna().sum()

print("Vor Zusammenführen war Spalte zu " + labels_en_completeness + " gefüllt")

#labels_en mit labels_tags auffüllen falls leer, mit labels auffüllen falls auch leer
df["labels_en"] = (df["labels_en"]
                                .fillna(df["labels_tags"]
                                .fillna(df["labels"])))

#Prozent und Anzahl der Spalte nach Bearbeitung
labels_en_completeness = str(df["labels_en"].notna().mean()*100)
labels_en_after = df["labels_en"].notna().sum()

labels_added = str(labels_en_after - labels_en_before)

print("Nach Zusammenführen ist Spalte zu " + labels_en_completeness + " gefüllt")
print("Es wurden " + labels_added + " Einträge hinzugefügt")

# labels und labels_tags löschen
df = df.drop(columns=["labels", "labels_tags"])

### Falsche Werte beheben
-  Nicht-englische in englisch umwandeln
-  'New' Eintrag gelöscht

In [ ]:
'''
Erstelle ein Wörterbuch, um die folgenden Labels zu übersetzen

'''

labels_dict = {
    'EG-Öko-Verordnung': 'eu organic',
    'Ohne Gentechnik': 'no gmos',
    'fr:eco-emballages': 'green dot',
    'pt:ecoponto-amarelo': 'yellow ecopoint',
    'Sistema de Etiquetado Frontal de Alimentos y Bebidas': 'front-of-pack nutrition labelling',
    'fr:Origine France': 'made in france', # to 'french origin'
    'fr:Sélection Intermarché': 'intermarché selection',
    'fr:Bleu Blanc Cœur': 'bleu blanc cœur'
}

zu_loeschen = r'\bNew\b'
df = delete_rows_with_words_in_spalte(df, "labels_en", zu_loeschen)

df['labels_en'] = df['labels_en'].apply(lambda x: uebersetze_mit_dict(x, labels_dict))

### 90 Prozent der labels Einträge zeigen
90% Prozent der Labels besteht aus 29 verschiedenen Einträgen

In [ ]:
# 1. Zählen und sortieren (Pandas macht das bei value_counts standardmäßig)
counts = df["labels_en"].str.split(",").explode().str.strip().value_counts()

# 2. & 3. Kumulierte Prozente berechnen
cum_percentage = counts.cumsum() / counts.sum()

# 4. Den Filter anwenden (Alle Labels, deren Wert <= 0.90 ist)
top_90_percent_labels = cum_percentage[cum_percentage <= 0.90]

# Das Ergebnis anzeigen
print(top_90_percent_labels)


# Brands
Sehr viele brands, einzeln zu bereinigen dauert zu lange, 90% der brands_en besteht aus 86720 verschiedenen Einträgen.

Funktion die brands_clean mit den top 1000 Marken und 'other' sonst enthält

### Brands_en auffüllen und löschen
-  Vollständigkeit von 62,7% auf 62,72% Prozent (+0.02%)
-  804 Einträge hinzugefügt in brands_en Spalte

In [ ]:
#Prozent und Anzahl der Spalte vor Bearbeitung
brands_en_completeness = str(df["brands_en"].notna().mean()*100)
brands_en_before = df["brands_en"].notna().sum()

print("Vor Zusammenführen war Spalte zu " + brands_en_completeness + " gefüllt")

#brands_en behalten, mit brands_tags auffüllen falls leer, mit brands auffüllen falls auch leer
df["brands_en"] = df["brands_en"].fillna(df["brands_tags"].fillna(df["brands"]))

#Prozent und Anzahl der Spalte nach Bearbeitung
brands_en_completeness = str(df["brands_en"].notna().mean()*100)
brands_en_after = df["brands_en"].notna().sum()

brands_en_added = str(brands_en_after - brands_en_before)

print("Nach Zusammenführen ist Spalte zu " + brands_en_completeness + " gefüllt")
print("Es wurden " + brands_en_added + " Einträge hinzugefügt")

df = df.drop(columns=["brands", "brands_tags"])

### Top 90% anschauen
Sehr viele brands, einzeln zu bereinigen dauert zu lange, 90% der brands_en besteht aus 86720 verschiedenen Einträgen

In [ ]:
top_90_brands = top_x_percent_spalte(df, "brands_en", 0.90)
print(top_90_brands)

### Mögliche Lösung

In [ ]:
# Erstellte eine neue Spalte "brands_clean", die die bereinigten Markennamen enthält
df = clean_brands_pandas(df, 1000)
brands_clean = df["brands_clean"]
brands_clean.sample(20)

In [ ]:
df['brands_en'] = df['brands_en'].str.lower().str.strip()
    
# 2. Akzente entfernen (é -> e, ä -> a)
# Wichtig: Benötigt oft das 'unidecode' Paket oder Pandas string normalize
df['brands_en'] = df['brands_en'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
    
# 3. Satzzeichen und Firmenzusätze (GmbH, Inc, SA) mit Regex entfernen
# Ersetzt alles was kein Buchstabe oder Zahl ist mit einem Leerzeichen
df['brands_en'] = df['brands_en'].str.replace(r'[^a-z0-9\s]', ' ', regex=True)
# Entfernt gängige Rechtsformen
df['brands_en'] = df['brands_en'].str.replace(r'\b(gmbh|inc|ltd|sa|co|kg|ag)\b', '', regex=True)
    
# Letzte doppelte Leerzeichen entfernen, die durch das Löschen entstanden sind
df['brands_en'] = df['brands_en'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Origins
-  80% besteht aus 76 Ländern
-  origins_en mit origins_tags und origins aufgefüllt falls fehlend
-  origins_en behalten, Rest gelöscht

### Origins anschauen

In [ ]:
origins_spalten = ["origins", "origins_tags", "origins_en"]
origins = df[origins_spalten]
origins.sample(20)

In [ ]:
contains_california = contains_x_in_spalte(df, "origins_en", "fr:porc-origine-france")
contains_california.sample(20)

### Top 80% anzeigen
-  besteht aus 76 Ländern

In [ ]:
top_90_origins = top_x_percent_spalte(df, "origins_en", 0.90)
print(top_90_origins)

to_fix = ['unspecified', 'unknown', 'european union and non european union', 'non european union', 'great britain',
          'north-east atlantic ocean', 'fr:lait-origine-france', 'fr:import', "provence-alpes-côte d'azur", 'provence',
          'fr:ecosse', "côte d'ivoire", 'nouvelle-aquitaine', 'california', 'south-western france', 'fr:hors-france',
          'fr:porc-origine-france', 'rhône-alpes', 'auvergne-rhône-alpes',  'alsace', 'es:agricultura-ue', 'world',
          'es:agricultura-no-ue', 'fr:hors-union-europeenne', 'south-america', 'camargue', 'fr:sud-ouest',
          'fr:agriculture-ue-non-ue', 'pays de la loire', 'québec', 'imported-unknown', 'auvergne', 'brittany', 'normandy',
          'northeast pacific ocean', 'bayern', 'northwest pacific ocean']

to_fix_to_80 = []

## Bereinigung
-  nur bis 80%, da dann wieder zu viele einzel Dinger

### Spalten auffüllen und reduzieren
-  5343 Einträge hinzugefügt
-  von 3,71% auf 3,83% (+0,12%)

In [ ]:
#Prozent und Anzahl der Spalte vor Bearbeitung
origins_en_completeness = str(df["origins_en"].notna().mean()*100)
origins_en_before = df["origins_en"].notna().sum()

print("Vor Zusammenführen war Spalte zu " + origins_en_completeness + " gefüllt")

#origins_en behalten, mit origins_tags auffüllen falls leer, mit origins auffüllen falls auch leer
df["origins_en"] = df["origins_en"].fillna(df["origins_tags"].fillna(df["origins"]))

#Prozent und Anzahl der Spalte nach Bearbeitung
origins_en_completeness = str(df["origins_en"].notna().mean()*100)
origins_en_after = df["origins_en"].notna().sum()

origins_en_added = str(origins_en_after - origins_en_before)

print("Nach Zusammenführen ist Spalte zu " + origins_en_completeness + " gefüllt")
print("Es wurden " + origins_en_added + " Einträge hinzugefügt")

df = df.drop(columns=["origins", "origins_tags"])

### Übersetzen und ersetzen bis zu 80%

In [ ]:
origins_mapping = {
    # --- Ursprüngliche Einträge ---
    'unspecified': None, 
    'unknown': None, 
    'european union and non european union': None, 
    'non european union': None, 
    'fr:import': None, 
    'fr:hors-france': None,
    'great britain': 'united kingdom', 
    'fr:ecosse': 'united kingdom',
    'fr:lait-origine-france': 'france', 
    'fr:porc-origine-france': 'france',
    "provence-alpes-côte d'azur": 'france', 
    'provence': 'france',
    'nouvelle-aquitaine': 'france', 
    'south-western france': 'france',
    'rhône-alpes': 'france', 
    'auvergne-rhône-alpes': 'france', 
    'alsace': 'france',
    'california': 'united states', 
    "côte d'ivoire": 'ivory coast',
    'es:agricultura-ue': 'european union', 
    'north-east atlantic ocean': 'atlantic ocean',
    
    # --- Vorherige Erweiterung ---
    'world': None,                                 
    'es:agricultura-no-ue': None,                  
    'fr:hors-union-europeenne': None,              
    'south-america': 'south america',              
    'camargue': 'france',                          
    'fr:sud-ouest': 'france',                      
    'fr:agriculture-ue-non-ue': None,              
    'pays de la loire': 'france',                  
    'québec': 'canada',                            
    'imported-unknown': None,                      
    'auvergne': 'france',
    
    # --- NEUESTE Einträge ---
    'brittany': 'france',                          # Region in Frankreich (Bretagne)
    'normandy': 'france',                          # Region in Frankreich (Normandie)
    'bayern': 'germany',                           # Bundesland in Deutschland
    'northeast pacific ocean': 'pacific ocean',    # Ozean
    'northwest pacific ocean': 'pacific ocean'     # Ozean
}

df["origins_en"] = df["origins_en"].apply(lambda x: uebersetze_mit_dict(x, origins_mapping))

# Food groups, pnns groups 1 und 2
-  ca 32% befüllt
-  pnns_groups jeweils mit food_groups_en befüllt falls fehlend
-  food_groups gelöscht
-  pnns_groups_1 vollständig sinnvolle Werte
-  pnns_groups_2 bis mind 90% sinnvole Werte

In [ ]:
pnns_groups_1_completeness = df["pnns_groups_1"].notna().mean()*100
print(pnns_groups_1_completeness)

food_groups_1_completeness = df["food_groups_en"].notna().mean()*100
print(food_groups_1_completeness)

### Inkonsistenz finden
-  meist nur leicht andere Schreibweise: zB 'Fats and Sauces' zu 'Fat and Sauces'

In [ ]:
def find_inconsistent_groups(df):
    # --- NEU: Harmonisierung bekannter OpenFoodFacts Tippfehler/Mappings ---
    # Wir passen die englische Spalte temporär an (z.B. Plural zu Singular), 
    # nutzen sie aber NUR für die Textvergleiche unten.
    food_groups_clean = df['food_groups_en'].astype(str)\
        .str.replace('Fats and sauces', 'Fat and sauces', regex=False)
        # .str.replace('Anderer Fehler', 'Richtiger Wert', regex=False) # <- Hier kannst du später weitere anketten

    # Fall 1: Echte NaNs (Hier nutzen wir die Originalspalte, da .astype(str) aus NaN "nan" macht)
    mask_nan = df['food_groups_en'].isna() & \
               (df['pnns_groups_1'] == 'unknown') & \
               (df['pnns_groups_2'] == 'unknown')

    # Fall 2: Beide PNNS Spalten sind gleich (Abgleich mit der harmonisierten Spalte)
    mask_duplicate = (df['pnns_groups_1'] == df['pnns_groups_2']) & \
                     (food_groups_clean == df['pnns_groups_1'].astype(str))

    # Fall 3: Der Standardfall -> "PNNS_1,PNNS_2" (Abgleich mit der harmonisierten Spalte)
    mask_concat = (food_groups_clean == df['pnns_groups_1'].astype(str) + ',' + df['pnns_groups_2'].astype(str))

    # Fall 4: Toleranter Abgleich (ohne Leerzeichen und Kommas)
    clean_food_str = food_groups_clean.str.replace(', ', '', regex=False).str.replace(',', '', regex=False).str.replace(' ', '', regex=False)
    clean_pnns_str = (df['pnns_groups_1'].astype(str) + df['pnns_groups_2'].astype(str)).str.replace(' ', '', regex=False)
    mask_fuzzy = clean_food_str == clean_pnns_str

    # Fall 5: Alle drei Spalten sind NaN
    mask_all_nan = df['food_groups_en'].isna() & \
                   df['pnns_groups_1'].isna() & \
                   df['pnns_groups_2'].isna()

    # Wir kombinieren alle akzeptierten Muster
    is_consistent = mask_nan | mask_duplicate | mask_concat | mask_fuzzy | mask_all_nan

    # Gibt alle Zeilen zurück, die immer noch abweichen
    return df[~is_consistent]

# Anwendung auf deinen DataFrame:
inconsistent_rows = find_inconsistent_groups(df)

# Ausgabe der Problemfälle, um neue Muster zu entdecken
print(f"Gefundene Abweichungen: {len(inconsistent_rows)}")
print(inconsistent_rows[['food_groups_en', 'pnns_groups_1', 'pnns_groups_2']].head(10))

In [ ]:
print(inconsistent_rows[['food_groups_en', 'pnns_groups_1', 'pnns_groups_2']].sample(10))

### Allgemein anschauen

In [ ]:
essen_spalten = ["food_groups_en", "pnns_groups_1", "pnns_groups_2"]
essen = df[essen_spalten]
essen.sample(20)


### Auffüllen
-  9120 Zeilen wurden aufgefüllt

In [ ]:
def fill_missing_pnns_groups(df):
    # 1. Maske definieren (Die erste, striktere Variante)
    mask_pnns_1 = df['pnns_groups_1'].isna() | (df['pnns_groups_1'] == 'unknown')
    mask_pnns_2 = df['pnns_groups_2'].isna() | (df['pnns_groups_2'] == 'unknown')
    mask_food = df['food_groups_en'].notna() & (df['food_groups_en'] != 'NaN')
    
    trigger_condition = mask_pnns_1 & mask_pnns_2 & mask_food

    # --- NEU: Zählen und Ausgeben der betroffenen Zeilen ---
    num_affected_rows = trigger_condition.sum()
    print(f"Info: {num_affected_rows} Zeilen wurden erfolgreich aufgefüllt.")

    # Falls keine Zeile zutrifft, direkt den DataFrame zurückgeben
    if not trigger_condition.any():
        return df

    # 2. Nur die betroffenen Zeilen extrahieren
    subset = df.loc[trigger_condition, 'food_groups_en'].astype(str)

    # 3. Am LETZTEN Komma aufteilen
    split_df = subset.str.rsplit(',', n=1, expand=True)

    if split_df.shape[1] == 1:
        split_df[1] = split_df[0]
    else:
        split_df[1] = split_df[1].fillna(split_df[0])

    # 4. Bereinigung nach deinem Beispiel
    pnns_1_clean = split_df[0].str.replace(',', '', regex=False).str.strip()
    pnns_2_clean = split_df[1].str.strip()

    # 5. Die bereinigten Werte zurückschreiben
    df.loc[trigger_condition, 'pnns_groups_1'] = pnns_1_clean
    df.loc[trigger_condition, 'pnns_groups_2'] = pnns_2_clean

    return df

# Anwendung auf deinen DataFrame:
df = fill_missing_pnns_groups(df)

## Food groups
-  wird gelöscht und davor hat es Zeilen in pnns_groups aufgefüllt

In [ ]:
df = df.drop(columns=["food_groups_en", "food_groups", "food_groups_tags"])

## Pnns group 1
-  besteht aus 13 Kategorien
-  muss 590 löschen (geht iwie nicht)

### Top 100% anschauen

In [ ]:
top_90_pnns_groups_1 = top_x_percent_spalte(df, "pnns_groups_1", 1)
top_90_pnns_groups_1

### Fehler beheben

In [ ]:
to_fix_pnns = ['590', 'fish, meat, eggs']
pnns_1_dict = {
    'fish, meat, eggs': 'fish meat eggs'
}

df["pnns_groups_1"] = df["pnns_groups_1"].apply(lambda x: uebersetze_mit_dict(x, pnns_1_dict))

In [ ]:
# 590 kommt in einem Eintrag vor, und dieser ist allgemein schlecht gepflegt, deshalb löschen wir diesen Eintrag komplett
# delete_rows_with_words_in_spalte(df, "pnns_groups_1", r'\b(?:590)\b')

## Pnns group 2
-  bis 90% auf jeden Fall sinnvolle und gute Kategorien

### Top 90% anschauen

In [ ]:
top_90_pnns_2 = top_x_percent_spalte(df, "pnns_groups_2", 1)
top_90_pnns_2

## strip und lowercase

In [ ]:
pnns_2_dict = {
    "fresse": "undefined"
}

df["pnns_groups_2"] = df["pnns_groups_2"].apply(lambda x: uebersetze_mit_dict(x, pnns_2_dict))

# Main Category
-  zu 41,06% befüllt
-  2 Einträge kamen mit Auffüllen dazu
-  main_category gelöscht
-  seehr viele verschiedene Einträge -> nicht so strukturiert wie pnns_groups
-  en: Präfix entfernt, "-" zu Leerzeichen, und lowercase verwandelt
-  main_category_clean mit 200 main_categories und other sonst

## Spalten auffüllen und löschen

In [ ]:
#Prozent und Anzahl der Spalte vor Bearbeitung
main_category_en_completeness = str(df["main_category_en"].notna().mean()*100)
main_category_en_before = df["main_category_en"].notna().sum()

print("Vor Zusammenführen war Spalte zu " + main_category_en_completeness + " gefüllt")

#main_category_en behalten, mit main_category auffüllen falls leer
df["main_category_en"] = df["main_category_en"].fillna(df["main_category"])

#Prozent und Anzahl der Spalte nach Bearbeitung
main_category_en_completeness = str(df["main_category_en"].notna().mean()*100)
main_category_en_after = df["main_category_en"].notna().sum()

main_category_en_added = str(main_category_en_after - main_category_en_before)

print("Nach Zusammenführen ist Spalte zu " + main_category_en_completeness + " gefüllt")
print("Es wurden " + main_category_en_added + " Einträge hinzugefügt")

df = df.drop(columns=["main_category"])

## Top 90%

In [ ]:
top_90_main_category = top_x_percent_spalte(df, "main_category_en", 0.90)
top_90_main_category

## Bereinigen

In [ ]:
def clean_main_category_pandas(df, top_n=200):
    # 1. Arbeite auf einer Kopie der Spalte oder überschreibe sie
    df_cat = df.copy()
    
    # 2. Präfixe (wie 'en:', 'fr:') am Anfang des Strings entfernen
    # ^[a-z]{2,3}: sucht nach 2-3 Buchstaben am Anfang (^), gefolgt von einem Doppelpunkt
    df["main_category_en"] = df['main_category_en'].str.replace(r'^en:', '', regex=True)
    df_cat['main_category_clean'] = df_cat['main_category_en'].str.replace(r'^en:', '', regex=True)
    
    # 3. Bindestriche durch Leerzeichen ersetzen, trimmen und in Kleinbuchstaben umwandeln
    df_cat['main_category_en'] = df['main_category_en'].str.replace('-', ' ').str.strip().str.lower()
    df_cat['main_category_clean'] = df_cat['main_category_clean'].str.replace('-', ' ').str.strip().str.lower()
    
    # 4. Thresholding: Den "Long Tail" in 'other' zusammenfassen
    # Behalte nur die Top N (z.B. Top 100) Kategorien, der Rest wird 'other'
    top_categories = df_cat['main_category_clean'].value_counts().nlargest(top_n).index
    df_cat.loc[~df_cat['main_category_clean'].isin(top_categories), 'main_category_clean'] = 'other'
    
    return df_cat

df = clean_main_category_pandas(df, 200)

# Purchase places
-  4,65% befüllt
-  mind 72% der vorhandenen Werte englische Namen von Ländern und Regionen

In [ ]:
purchase_places_completeness = df["purchase_places"].notna().mean()*100
purchase_places_completeness

## Top 90%

In [ ]:
top_90_purchase_places = top_x_percent_spalte(df, "purchase_places", 0.9)
top_90_purchase_places


## Bereinigen

In [ ]:
to_fix_purchase = ["deutschland", "españa", "italia", "belgique", "suisse", "česká republika", "polska", "česko",
                   "usa", "méxico", "sverige", "normandie", "bayern", "praha", "tunisie", "münster", "nederland",
                   "österreich", "uk", "россия"]

purchase_places_mapping = {
    "deutschland": "germany",
    "españa": "spain",
    "italia": "italy",
    "belgique": "belgium",
    "suisse": "switzerland",
    "česká republika": "czech republic",
    "polska": "poland",
    "česko": "czech republic",
    "usa": "united states",
    "méxico": "mexico",
    "sverige": "sweden",
    
    # Regionen und Städte beibehalten (auf Englisch übersetzt)
    "normandie": "normandy",     # Region in Frankreich
    "bayern": "bavaria",         # Bundesland in Deutschland
    "praha": "prague",           # Stadt (Prag)
    "münster": "munster",        # Stadt in Deutschland
    
    "tunisie": "tunisia",
    "nederland": "netherlands",
    "österreich": "austria",
    "uk": "united kingdom",
    "россия": "russia"           # Russisch für Russland
}

df["purchase_places"] = df["purchase_places"].apply(lambda x: uebersetze_mit_dict(x, purchase_places_mapping))

# Brand owner
-  7,06% befüllt
-  Spalte wird so belassen (fehlendes Wissen, was fehlerhaft)

## Anschauen

In [ ]:
brand_owner_completeness = df["brand_owner"].notna().mean()*100
brand_owner_completeness

In [ ]:
top_90_brand_owner = top_x_percent_spalte(df, "brand_owner", 0.9)
top_90_brand_owner

In [ ]:
brands_dict = {'dummy_value' : 'strip und lower'}

df["brand_owner"] = df["brand_owner"].apply(lambda x: uebersetze_mit_dict(x, brands_dict))

# Popularity tags
-  34,93% befüllt
-  "-" mit Leerzeichen ersetzt

## Anschauen

In [ ]:
popularity_tags_completeness = df["popularity_tags"].notna().mean()*100
popularity_tags_completeness

In [ ]:
top_90_popularity = top_x_percent_spalte(df, "popularity_tags", 0.9)
top_90_popularity

## Bereinigen
-  Ersetze "-" mit Leerzeichen
- strip und lower

In [ ]:
df["popularity_tags"] = df["popularity_tags"].str.replace('-', ' ').str.strip().str.lower()

# No nutrition data
-  Werte: "on", "off", ""true", "false"
-  0,26% befüllt
-  Vorschlag: löschen

## Anschauen

In [ ]:
no_nutrition_completeness = df["no_nutrition_data"].notna().mean()*100
no_nutrition_completeness

In [ ]:
top_90_no_nutrition = top_x_percent_spalte(df, "no_nutrition_data", 1)
top_90_no_nutrition

## Bereinigen

In [ ]:
df["no_nutrition_data"] = df["no_nutrition_data"].str.strip().str.lower()

# Cities tags

## Anschauen

In [ ]:
cities_tags_completeness = completeness_spalte(df, "cities_tags")
cities_tags_completeness

In [ ]:
top_90_cities_tags = top_x_percent_spalte(df, "cities_tags", 0.9)
top_90_cities_tags

## Bereinigen

In [ ]:
df["cities_tags"] = df["cities_tags"].str.replace("-", " ").str.strip().str.lower()

# Suche

In [ ]:
df_filter = df[df["pnns_groups_1"].str.contains(r"590", na=False, case=False, regex=True)]
print(df_filter.sample(1))

In [ ]:
random_10 = df.sample(10)
random_10

# Entfernung von Duplikaten in einer Zelle

In [ ]:
def remove_duplicates(text):
    # Überspringe fehlende Werte (NaN)
    if pd.isna(text):
        return text
    
    # 1. Am Komma trennen und überflüssige Leerzeichen (strip) entfernen
    items = [item.strip() for item in str(text).split(',')]
    
    # 2. Duplikate entfernen (dict.fromkeys erhält im Gegensatz zu set() die ursprüngliche Reihenfolge)
    unique_items = list(dict.fromkeys(items))
    
    # 3. Wieder zu einem sauberen String zusammensetzen
    return ', '.join(unique_items)


#for spalte in spalten:
    df[spalte] = df[spalte].apply(remove_duplicates)
